# Deep Q-Learning——Lunar Lander

本作业将训练一个 Agent，让月球着陆器安全降落到月面上的指定平台。

## 目录

- [1 - 导入软件包](#1)
- [2 - Hyperparameters](#2)
- [3 - Lunar Lander 环境](#3)
  - [3.1 Action Space](#3.1)
  - [3.2 Observation Space](#3.2)
  - [3.3 Rewards](#3.3)
  - [3.4 Episode 结束条件](#3.4)
- [4 - 加载环境](#4)
- [5 - 与 Gym 环境交互](#5)
  - [5.1 探索环境的状态变化](#5.1)
- [6 - Deep Q-Learning](#6)
  - [6.1 Target Network](#6.1)
    - [练习 1](#ex01)
  - [6.2 Experience Replay](#6.2)
- [7 - 结合 Experience Replay 的 Deep Q-Learning 算法](#7)
  - [练习 2](#ex02)
- [8 - 更新网络权重](#8)
- [9 - 训练 Agent](#9)
- [10 - 查看训练后的 Agent](#10)
- [11 - 恭喜完成](#11)
- [12 - 参考文献](#12)

<img src="./images/lunar_lander.gif" style="width:60%;" alt="Lunar Lander 演示">


> **注意：** 为避免 autograder（自动评分器）报错，提交作业时请勿编辑或删除不参与评分的单元，也不要新增单元。
>
> **通过作业后**，如果希望尝试修改不参与评分的代码，可以按照本 Notebook 底部的说明操作。


<a name="1"></a>
## 1 - 导入软件包

本实验使用以下软件包与工具：

- `numpy`：Python 科学计算库。
- `deque`：用作 Memory Buffer 的数据结构。
- `namedtuple`：保存 Experience Tuples。
- `gym`：提供用于测试 Reinforcement Learning 算法的环境。本 Notebook 使用 `gym 0.24.0`。
- `PIL.Image` 与 `pyvirtualdisplay`：用于渲染 Lunar Lander 环境。
- `tensorflow.keras` 的多个模块：用于构建 Deep Learning 模型。
- `utils`：包含本作业的辅助函数，无需修改。

运行下面的单元导入全部依赖。


In [ ]:
import time
from collections import deque, namedtuple

import gym
import numpy as np
import PIL.Image
import tensorflow as tf
import utils

from pyvirtualdisplay import Display
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense, Input
from tensorflow.keras.losses import MSE
from tensorflow.keras.optimizers import Adam

In [ ]:
# Set up a virtual display to render the Lunar Lander environment.
Display(visible=0, size=(840, 480)).start();

# Set the random seed for TensorFlow
tf.random.set_seed(utils.SEED)

<a name="2"></a>
## 2 - Hyperparameters

运行下面的单元设置 Hyperparameters。


In [ ]:
MEMORY_SIZE = 100_000     # size of memory buffer
GAMMA = 0.995             # discount factor
ALPHA = 1e-3              # learning rate  
NUM_STEPS_FOR_UPDATE = 4  # perform a learning update every C time steps

<a name="3"></a>
## 3 - Lunar Lander 环境

本 Notebook 使用 [OpenAI Gym](https://www.gymlibrary.dev/)，它为 Reinforcement Learning 提供多种环境。Environment 表示需要解决的问题或任务；本实验的任务是 Lunar Lander。

目标是让着陆器安全降落到月面平台。两面旗帜标出平台，中心坐标为 `(0, 0)`；环境也允许在平台之外着陆。着陆器从环境顶部中央出发，其质心会受到随机初始力，燃料无限。本作业以最近 100 个 Episode 的平均得分达到 200 作为解决环境的标准。

<figure style="text-align:center;">
<img src="images/lunar_lander.gif" style="width:40%;" alt="Lunar Lander 环境">
<figcaption>图 1：Lunar Lander 环境。</figcaption>
</figure>

<a name="3.1"></a>
### 3.1 Action Space

Agent 有四个离散动作，代码中的编号如下：

| 编号 | 动作 |
| --- | --- |
| 0 | 不执行操作 |
| 1 | 启动右侧发动机 |
| 2 | 启动主发动机 |
| 3 | 启动左侧发动机 |

<a name="3.2"></a>
### 3.2 Observation Space

State Vector 包含 8 个变量：位置 $(x,y)$、线速度 $(\dot x,\dot y)$、角度 $\theta$、角速度 $\dot\theta$，以及表示左右支腿是否接触地面的两个布尔值 $l$、$r$。平台中心始终对应 $(0,0)$。

<a name="3.3"></a>
### 3.3 Rewards

每一步都会产生 Reward，一个 Episode 的总 Reward 为各步 Reward 之和。奖励设计考虑以下因素：

- 越接近平台，奖励越高；远离平台则降低。
- 移动越慢，奖励越高；越快则降低。
- 倾斜越严重，奖励越低。
- 每条支腿接触地面对应 10 分的奖励项。
- 侧发动机每启动一帧扣 0.03 分，主发动机每启动一帧扣 0.3 分。

坠毁或安全着陆时，额外获得 -100 或 +100 分。

<a name="3.4"></a>
### 3.4 Episode 结束条件

结束条件包括着陆器机身接触月面而坠毁，以及 $x$ 坐标的绝对值超过 1，即越过左右边界。完整环境说明可参见 [Gym 文档](https://www.gymlibrary.dev/environments/box2d/lunar_lander/)。


<a name="4"></a>
## 4 - 加载环境

通过 `gym.make()` 加载本作业使用的 `LunarLander-v2`。该环境的版本历史可参见 [Gym 文档](https://www.gymlibrary.dev/environments/box2d/lunar_lander/#version-history)。


In [ ]:
env = gym.make('LunarLander-v2')

加载后，调用 `.reset()` 将环境重置到初始状态。着陆器从顶部中央出发，可通过 `.render()` 渲染第一帧。


In [ ]:
env.reset()
PIL.Image.fromarray(env.render(mode='rgb_array'))

构建 Neural Network 前，需要确定 State Vector 的大小与可用动作数量。分别通过环境的 `.observation_space.shape` 和 `.action_space.n` 属性获取。


In [ ]:
state_size = env.observation_space.shape
num_actions = env.action_space.n

print('State Shape:', state_size)
print('Number of actions:', num_actions)

<a name="5"></a>
## 5 - 与 Gym 环境交互

Gym 实现了标准的 Agent–Environment Loop：

<figure style="text-align:center;">
<video src="./videos/rl_formalism.m4v" width="840" height="480" controls autoplay loop poster="./images/rl_formalism.png"></video>
<figcaption>图 2：Agent–Environment Loop。</figcaption>
</figure>

Agent 在离散时间步 $t=0,1,2,\ldots$ 与环境交互。在时间步 $t$，它观察状态 $S_t$，根据 Policy $\pi$ 选择动作 $A_t$；执行后得到数值 Reward $R_t$，并转移到下一状态 $S_{t+1}$。

<a name="5.1"></a>
### 5.1 探索环境的状态变化

调用 `.step()` 推进一个时间步。在本 Notebook 使用的 Gym 版本中，该方法接收一个动作并返回四项：

- `observation`（object）：环境相关的观测。本例是包含位置、速度等信息的 NumPy 数组，详见 [3.2 Observation Space](#3.2)。
- `reward`（float）：执行动作获得的奖励，本例为 `numpy.float64`，详见 [3.3 Rewards](#3.3)。
- `done`（boolean）：为 `True` 时表示 Episode 已结束，需要重置环境。
- `info`（dictionary）：用于调试的诊断信息，本 Notebook 不使用它。

开始一个 Episode 前，先调用 `.reset()` 获取初始状态。


In [ ]:
# Reset the environment and get the initial state.
current_state = env.reset()

重置环境后，Agent 可通过 `.step()` 执行动作，每个时间步只能选择一个。下面可以尝试不同动作，观察返回值如何变化。

动作编号为：`0` 不操作，`1` 启动右侧发动机，`2` 启动主发动机，`3` 启动左侧发动机。


In [ ]:
# Select an action
action = 0

# Run a single time step of the environment's dynamics with the given action.
next_state, reward, done, _ = env.step(action)

# Display table with values.
utils.display_table(current_state, action, next_state, reward, done)

# Replace the `current_state` with the state after the action is taken
current_state = next_state

实际训练时，使用循环让 Agent 在一个 Episode 中连续执行多个动作。


<a name="6"></a>
## 6 - Deep Q-Learning

当 State Space 和 Action Space 都离散且规模可控时，可以利用 Bellman Equation 迭代估计 Action-value Function。对一次状态转移，可用以下形式理解更新目标：

$$Q_{i+1}(s,a)=R+\gamma\max_{a'}Q_i(s',a').$$

在满足相应收敛条件时，表格形式的方法可逐步逼近最优 Action-value Function $Q^*(s,a)$。但对于连续 State Space，无法逐一访问并保存全部 State–Action 组合的估计值。

Deep Q-Learning 使用 Neural Network 近似 $Q^*(s,a)$，这个网络称为 Q-Network。通过调整权重，使网络预测与 Bellman Target 之间的 Mean Squared Error 尽可能小。

直接用 Neural Network 估计 Action Values 容易造成训练不稳定。下面介绍两种改善稳定性的技术：**Target Network** 和 **Experience Replay**。


<a name="6.1"></a>
### 6.1 Target Network

如果直接使用当前 Q-Network 生成 Target，在非终止状态下有：

$$y=R+\gamma\max_{a'}Q(s',a';w),$$

其中 $w$ 为网络权重。训练时最小化的 Squared Error 来自：

$$\overbrace{\underbrace{R+\gamma\max_{a'}Q(s',a';w)}_{y\text{ target}}-Q(s,a;w)}^{\text{Error}}.$$

由于预测值和 Target 都依赖同一组权重，每轮更新都会改变 Target，可能产生振荡与不稳定。为此，创建另一个结构相同的 **Target Q-Network** $\hat Q$，专门生成 Target：

$$\overbrace{\underbrace{R+\gamma\max_{a'}\hat Q(s',a';w^-)}_{y\text{ target}}-Q(s,a;w)}^{\text{Error}}.$$

$w^-$ 和 $w$ 分别是 Target Network 与 Q-Network 的权重。本实验每隔 $C$ 个时间步、且 Replay Buffer 满足条件时进行一次学习更新，用 Target Network 计算目标，并对其权重执行 Soft Update：

$$w^-\leftarrow\tau w+(1-\tau)w^-,$$

其中 $\tau\ll1$。这样 Target 的变化较慢，有助于改善学习过程的稳定性。


<a name="ex01"></a>
### 练习 1

创建 Q-Network、Target Q-Network，并设置 Optimizer。Deep Q-Network（DQN）通过学习从 State 到各动作 Q 值的映射，近似最优 Action-value Function。

两个网络均采用以下架构：

- `Input`：输入形状为 `state_size`。
- `Dense`：64 个单元，使用 `relu`。
- `Dense`：64 个单元，使用 `relu`。
- Output Layer 为 `Dense`：`num_actions` 个单元，使用 `linear`。

Optimizer 设为 `Adam`，Learning Rate 使用 [Hyperparameters](#2) 部分定义的 `ALPHA`。本练习应使用已经导入的组件：

```python
from tensorflow.keras.layers import Dense, Input
from tensorflow.keras.optimizers import Adam
```


In [ ]:
# UNQ_C1
# GRADED CELL

# Create the Q-Network
q_network = Sequential([
    ### START CODE HERE ### 
    
    
    
    
    ### END CODE HERE ### 
    ])

# Create the target Q^-Network
target_q_network = Sequential([
    ### START CODE HERE ### 
    
    
    
    
    ### END CODE HERE ###
    ])

### START CODE HERE ### 
optimizer = None
### END CODE HERE ###

In [ ]:
# UNIT TEST
from public_tests import *

test_network(q_network)
test_network(target_q_network)
test_optimizer(optimizer, ALPHA) 

<details>
<summary><strong>点击查看提示</strong></summary>

```python
# Create the Q-Network
q_network = Sequential([
    Input(shape=state_size),                      
    Dense(units=64, activation='relu'),            
    Dense(units=64, activation='relu'),            
    Dense(units=num_actions, activation='linear'),
    ])

# Create the target Q^-Network
target_q_network = Sequential([
    Input(shape=state_size),                       
    Dense(units=64, activation='relu'),            
    Dense(units=64, activation='relu'),            
    Dense(units=num_actions, activation='linear'), 
    ])

optimizer = Adam(learning_rate=ALPHA)
```

</details>


<a name="6.2"></a>
### 6.2 Experience Replay

Agent 与环境交互得到的状态、动作和奖励天然具有时序关系。直接从连续的 Experiences 学习，可能受到强相关性的影响。

**Experience Replay** 将 Experiences 保存到 Memory Buffer，再随机抽取一个 Mini-batch 进行学习，以减弱样本间的相关性。每个时间步都会保存转移 $(S_t,A_t,R_t,S_{t+1})$；本实现还保存 Episode 是否结束的 `done` 标记。

为方便访问各字段，使用 Named Tuples 保存 Experiences。


In [ ]:
# Store experiences as named tuples
experience = namedtuple("Experience", field_names=["state", "action", "reward", "next_state", "done"])

Experience Replay 有助于减轻相关性带来的振荡和不稳定，也允许同一条 Experience 参与多次权重更新，提高 Data Efficiency。


<a name="7"></a>
## 7 - 结合 Experience Replay 的 Deep Q-Learning 算法

将前面介绍的方法组合起来，得到如下算法：

<figure style="text-align:center;">
<img src="images/deep_q_algorithm.png" style="width:90%;border:thin silver solid;padding:0;" alt="结合 Experience Replay 的 Deep Q-Learning 算法">
<figcaption>图 3：结合 Experience Replay 的 Deep Q-Learning。</figcaption>
</figure>


<a name="ex02"></a>
### 练习 2

实现图 3 算法的第 12 行，并计算 Target 与 $Q(s,a)$ 之间的 Loss。完成 `compute_loss`，设置：

$$y_j=\begin{cases}
R_j,&\text{Episode terminates at step }j+1,\\
R_j+\gamma\max_{a'}\hat Q(s_{j+1},a'),&\text{otherwise}.
\end{cases}$$

注意以下几点：

- 函数输入是一个 Experience Mini-batch，解包后得到 `states`、`actions`、`rewards`、`next_states` 和 `done_vals`。它们是 TensorFlow Tensors，大小取决于 Batch Size。例如，Batch Size 为 64 时，`rewards` 与 `done_vals` 各含 64 个元素。
- 对含多个元素的 Tensor，不能直接用普通 `if/else` 为每个元素设置 Target。可以用逐元素运算：`done_vals` 中的终止标记转换为数值后为 1，未终止为 0，因此 `(1 - done_vals)` 会在终止样本上将未来奖励项置零。

最后计算 `y_targets` 与 `q_values` 的 Mean Squared Error，使用已经导入的 `MSE`：

```python
from tensorflow.keras.losses import MSE
```


In [ ]:
# UNQ_C2
# GRADED FUNCTION: calculate_loss

def compute_loss(experiences, gamma, q_network, target_q_network):
    """ 
    Calculates the loss.
    
    Args:
      experiences: (tuple) tuple of ["state", "action", "reward", "next_state", "done"] namedtuples
      gamma: (float) The discount factor.
      q_network: (tf.keras.Sequential) Keras model for predicting the q_values
      target_q_network: (tf.keras.Sequential) Keras model for predicting the targets
          
    Returns:
      loss: (TensorFlow Tensor(shape=(0,), dtype=int32)) the Mean-Squared Error between
            the y targets and the Q(s,a) values.
    """

    # Unpack the mini-batch of experience tuples
    states, actions, rewards, next_states, done_vals = experiences
    
    # Compute max Q^(s,a)
    max_qsa = tf.reduce_max(target_q_network(next_states), axis=-1)
    
    # Set y = R if episode terminates, otherwise set y = R + γ max Q^(s,a).
    ### START CODE HERE ### 
    y_targets = None
    ### END CODE HERE ###
    
    # Get the q_values and reshape to match y_targets
    q_values = q_network(states)
    q_values = tf.gather_nd(q_values, tf.stack([tf.range(q_values.shape[0]),
                                                tf.cast(actions, tf.int32)], axis=1))
        
    # Compute the loss
    ### START CODE HERE ### 
    loss = None 
    ### END CODE HERE ### 
    
    return loss

In [ ]:
# UNIT TEST    
test_compute_loss(compute_loss)

<details>
<summary><strong>点击查看提示</strong></summary>

```python
def compute_loss(experiences, gamma, q_network, target_q_network):
    """ 
    Calculates the loss.

    Args:
      experiences: (tuple) tuple of ["state", "action", "reward", "next_state", "done"] namedtuples
      gamma: (float) The discount factor.
      q_network: (tf.keras.Sequential) Keras model for predicting the q_values
      target_q_network: (tf.keras.Sequential) Keras model for predicting the targets

    Returns:
      loss: (TensorFlow Tensor(shape=(0,), dtype=int32)) the Mean-Squared Error between
            the y targets and the Q(s,a) values.
    """


    # Unpack the mini-batch of experience tuples
    states, actions, rewards, next_states, done_vals = experiences

    # Compute max Q^(s,a)
    max_qsa = tf.reduce_max(target_q_network(next_states), axis=-1)

    # Set y = R if episode terminates, otherwise set y = R + γ max Q^(s,a).
    y_targets = rewards + (gamma * max_qsa * (1 - done_vals))

    # Get the q_values
    q_values = q_network(states)
    q_values = tf.gather_nd(q_values, tf.stack([tf.range(q_values.shape[0]),
                                                tf.cast(actions, tf.int32)], axis=1))

    # Calculate the loss
    loss = MSE(y_targets, q_values)

    return loss
```

</details>


<a name="8"></a>
## 8 - 更新网络权重

下面的 `agent_learn` 实现[图 3](#7) 的第 12–14 行，通过 Custom Training Loop 更新 Q-Network 和 Target Q-Network。使用 `tf.GradientTape` 计算 Gradients，再调用 `optimizer.apply_gradients()` 更新 Q-Network 权重。

代码还使用 `@tf.function` 改善性能；原实验环境中，未使用该装饰器时训练耗时约为两倍，实际效果取决于运行环境。更多信息可参见 [TensorFlow 文档](https://www.tensorflow.org/guide/function)。

函数最后通过 [Soft Update](#6.1) 更新 Target Network。具体实现可查看 `utils.update_target_network`。


In [ ]:
@tf.function
def agent_learn(experiences, gamma):
    """
    Updates the weights of the Q networks.
    
    Args:
      experiences: (tuple) tuple of ["state", "action", "reward", "next_state", "done"] namedtuples
      gamma: (float) The discount factor.
    
    """
    
    # Calculate the loss
    with tf.GradientTape() as tape:
        loss = compute_loss(experiences, gamma, q_network, target_q_network)

    # Get the gradients of the loss with respect to the weights.
    gradients = tape.gradient(loss, q_network.trainable_variables)
    
    # Update the weights of the q_network.
    optimizer.apply_gradients(zip(gradients, q_network.trainable_variables))

    # update the weights of target q_network
    utils.update_target_network(q_network, target_q_network)

<a name="9"></a>
## 9 - 训练 Agent

现在开始训练 Agent，解决 Lunar Lander 任务。下面的代码对应[图 3](#7) 的各行；本节末尾再次展示算法，方便对照。

- **第 1 行：** 初始化容量为 $N=$ `MEMORY_SIZE` 的 `memory_buffer`，数据结构使用 `deque`。
- **第 2 行：** 跳过此步，因为已在[练习 1](#ex01) 中创建 `q_network`。
- **第 3 行：** 将 `target_q_network` 的权重初始化为与 `q_network` 相同。
- **第 4 行：** 开始外层循环，设置 $M=$ `num_episodes=2000`。使用默认参数时，预期可在此上限内解决环境。
- **第 5 行：** 调用 `.reset()`，重置环境并获取初始状态。
- **第 6 行：** 开始内层循环，设置 $T=$ `max_num_timesteps=1000`。如果 Episode 尚未结束，达到此上限也会退出当前 Episode 的循环。
- **第 7 行：** 根据当前 `state`，使用 $\epsilon$-greedy Policy 选择 `action`。初始 `epsilon=1`，对应等概率随机选择动作。随着训练进行，按给定 Decay Rate 逐渐减小 $\epsilon$，使 Agent 更倾向于选择它认为能最大化 $Q(s,a)$ 的动作。最小值设为 `0.01`，保留少量 Exploration；若为 0，则成为纯 Greedy Policy。具体实现见 `utils.get_action`。
- **第 8 行：** 调用 `.step(action)`，获得 `reward` 与 `next_state` 等返回值。
- **第 9 行：** 将 `experience(state, action, reward, next_state, done)` 保存到 Buffer。`done` 用于标记 Episode 结束，也用于[练习 2](#ex02) 的 Target 计算。
- **第 10 行：** 调用 `utils.check_update_conditions` 检查是否更新：是否已到每 $C=$ `NUM_STEPS_FOR_UPDATE=4` 步的更新时机，以及 Buffer 是否有足够的样本。例如，Batch Size 为 64 时，本实现要求 Buffer 中超过 64 条 Experience。满足条件返回 `True`，否则返回 `False`。
- **第 11–14 行：** 如果 `update` 为 `True`，随机抽取 Experience Mini-batch，计算 Target，执行 Gradient Descent，并更新两个网络。后三步由[第 8 节](#8)的 `agent_learn` 完成。
- **第 15 行：** 将 `next_state` 设为新的 `state`，并检查 `done`；若为 `True`，退出内层循环。
- **第 16 行：** 每个 Episode 结束后更新 $\epsilon$，并检查最近 100 个 Episode 的平均得分是否达到 200。若达到，视为解决环境；否则继续下一轮。

代码还记录每个 Episode 的总得分，用于判断是否完成任务及绘制学习过程，并用 `time` 测量训练耗时。

<figure style="text-align:center;">
<img src="images/deep_q_algorithm.png" style="width:90%;border:thin silver solid;padding:0;" alt="训练循环对应的 Deep Q-Learning 算法">
<figcaption>图 4：结合 Experience Replay 的 Deep Q-Learning。</figcaption>
</figure>

**注意：** 使用默认参数时，原实验环境中下面的单元约需运行 10–15 分钟，实际耗时取决于硬件与运行环境。


In [ ]:
start = time.time()

num_episodes = 2000
max_num_timesteps = 1000

total_point_history = []

num_p_av = 100    # number of total points to use for averaging
epsilon = 1.0     # initial ε value for ε-greedy policy

# Create a memory buffer D with capacity N
memory_buffer = deque(maxlen=MEMORY_SIZE)

# Set the target network weights equal to the Q-Network weights
target_q_network.set_weights(q_network.get_weights())

for i in range(num_episodes):
    
    # Reset the environment to the initial state and get the initial state
    state = env.reset()
    total_points = 0
    
    for t in range(max_num_timesteps):
        
        # From the current state S choose an action A using an ε-greedy policy
        state_qn = np.expand_dims(state, axis=0)  # state needs to be the right shape for the q_network
        q_values = q_network(state_qn)
        action = utils.get_action(q_values, epsilon)
        
        # Take action A and receive reward R and the next state S'
        next_state, reward, done, _ = env.step(action)
        
        # Store experience tuple (S,A,R,S') in the memory buffer.
        # We store the done variable as well for convenience.
        memory_buffer.append(experience(state, action, reward, next_state, done))
        
        # Only update the network every NUM_STEPS_FOR_UPDATE time steps.
        update = utils.check_update_conditions(t, NUM_STEPS_FOR_UPDATE, memory_buffer)
        
        if update:
            # Sample random mini-batch of experience tuples (S,A,R,S') from D
            experiences = utils.get_experiences(memory_buffer)
            
            # Set the y targets, perform a gradient descent step,
            # and update the network weights.
            agent_learn(experiences, GAMMA)
        
        state = next_state.copy()
        total_points += reward
        
        if done:
            break
            
    total_point_history.append(total_points)
    av_latest_points = np.mean(total_point_history[-num_p_av:])
    
    # Update the ε value
    epsilon = utils.get_new_eps(epsilon)

    print(f"\rEpisode {i+1} | Total point average of the last {num_p_av} episodes: {av_latest_points:.2f}", end="")

    if (i+1) % num_p_av == 0:
        print(f"\rEpisode {i+1} | Total point average of the last {num_p_av} episodes: {av_latest_points:.2f}")

    # We will consider that the environment is solved if we get an
    # average of 200 points in the last 100 episodes.
    if av_latest_points >= 200.0:
        print(f"\n\nEnvironment solved in {i+1} episodes!")
        q_network.save('lunar_lander_model.h5')
        break
        
tot_time = time.time() - start

print(f"\nTotal Runtime: {tot_time:.2f} s ({(tot_time/60):.2f} min)")

绘制各 Episode 的总得分及 Moving Average，可以观察 Agent 在训练中的进步。更多绘图选项可查看 `utils.plot_history` 的实现。


In [ ]:
# Plot the total point history along with the moving average
utils.plot_history(total_point_history)

<a name="10"></a>
## 10 - 查看训练后的 Agent

训练完成后，使用 `utils.create_video` 生成 Agent 与环境交互的视频，动作由训练后的 Q-Network 选择。该函数使用 `imageio` 创建视频；下面先设置以屏蔽其部分非关键警告。


In [ ]:
# Suppress warnings from imageio
import logging
logging.getLogger().setLevel(logging.ERROR)

下面使用训练后的 `q_network` 生成 Lunar Lander 演示视频，按指定 `filename` 保存到 `videos` 目录，再通过 `utils.embed_mp4` 嵌入 Jupyter Notebook，便于直接观看。

由于每次开始时施加的初始力是随机的，多次运行可能得到不同视频。训练良好的 Agent 应能适应不同初始扰动，并较稳定地降落到平台上。


In [ ]:
filename = "./videos/lunar_lander.mp4"

utils.create_video(filename, env, q_network)
utils.embed_mp4(filename)

<a name="11"></a>
## 11 - 恭喜完成！

你已经结合 Deep Q-Learning 与 Experience Replay，训练 Agent 控制月球着陆器安全降落到月面平台。


<a name="12"></a>
## 12 - 参考文献

如需进一步了解 Deep Q-Learning，可以阅读以下论文：

- Mnih, V., Kavukcuoglu, K., Silver, D. et al. *Human-level control through deep reinforcement learning*. Nature 518, 529–533 (2015).
- Lillicrap, T. P., Hunt, J. J., Pritzel, A. et al. *Continuous Control with Deep Reinforcement Learning*. ICLR (2016).
- Mnih, V., Kavukcuoglu, K., Silver, D. et al. *Playing Atari with Deep Reinforcement Learning*. arXiv:1312.5602 (2013).


<details>
<summary><span style="color:darkgreen;"><strong>如需尝试修改不参与评分的代码，请点击这里。</strong></span></summary>

**重要提示：请在通过作业后再执行这些操作，以免影响 autograder。**

1. 在 Notebook 菜单中，依次选择 `View` > `Cell Toolbar` > `Edit Metadata`。
2. 在希望锁定或解锁的代码单元旁，点击 `Edit Metadata`。
3. 将 `editable` 属性设为 `true` 以解锁，或设为 `false` 以锁定。
4. 在 Notebook 菜单中，依次选择 `View` > `Cell Toolbar` > `None`。

下面的简短演示展示了上述步骤：

<img src="https://lh3.google.com/u/0/d/14Xy_Mb17CZVgzVAgq7NCjMVBvSae3xO1" alt="解锁 Notebook 单元的操作演示（unlock_cells.gif）">

</details>
